# Notebook 104 — Evaluación con MLflow

Tienes un agente que responde. La pregunta ahora es: **¿responde bien?** Y más importante:
¿cómo lo sabes de forma que puedas defenderlo con datos?

Este notebook monta dos capas de evaluación complementarias:

| Capa | Qué mide | Costo | Cuándo falla |
|---|---|---|---|
| **A — determinística** | Citas contra `gold_chunk_ids` | Cero llamadas al LLM | Nunca (es aritmética) |
| **B — LLM judges** | Correctitud, groundedness, relevancia | Una llamada por scorer por pregunta | Bajo throttling |

La capa A replica la métrica de la competencia. La capa B captura lo que ninguna fórmula
puede: si la respuesta *significa* lo mismo que la gold aunque esté redactada distinto, y si
está realmente respaldada por el contexto recuperado.

Corremos las capas por separado, a propósito: si los judges fallan por límites de cuota, la
capa A ya quedó registrada y no pierdes la corrida.

## 1. Cargar el agente

`%run` ejecuta el notebook 103 y deja sus funciones disponibles aquí: `rag_agent`,
`retrieve`, `generate`, las constantes y el experimento de MLflow ya configurado. Evita
duplicar código entre notebooks.

In [0]:
%run ./103_rag_agent

In [0]:
dbutils.widgets.text("n_eval", "20", "Número de preguntas a evaluar")
N_EVAL = int(dbutils.widgets.get("n_eval"))

print(f"Preguntas a evaluar : {N_EVAL}")
print(f"Experimento         : {EXPERIMENT_PATH}")
print(f"LLM                 : {LLM_ENDPOINT}")

## 2. Preparar el dataset de evaluación

MLflow espera una estructura concreta:

- **`inputs`** — un dict cuyas claves coinciden con los parámetros de la función que evalúas.
  Como nuestro agente recibe `question`, la clave debe llamarse `question`.
- **`expectations`** — la verdad de referencia. `expected_response` es una clave que los
  scorers integrados reconocen; `gold_chunk_ids` es nuestra, y la leen los scorers custom.

In [0]:
dev_sample = spark.table(T_DEV).orderBy("question_id").limit(N_EVAL).collect()

eval_data = [
    {
        "inputs": {"question": r["question"]},
        "expectations": {
            "expected_response": r["answer"],
            "gold_chunk_ids": list(r["gold_chunk_ids"]),
        },
    }
    for r in dev_sample
]

print(f"Dataset de evaluación: {len(eval_data)} casos\n")
print("Ejemplo:")
print(f"  pregunta : {eval_data[0]['inputs']['question']}")
print(f"  esperada : {eval_data[0]['expectations']['expected_response']}")
print(f"  gold     : {eval_data[0]['expectations']['gold_chunk_ids']}")

## 3. Capa A — Scorers de la competencia

Un scorer es una función decorada con `@scorer` que MLflow ejecuta sobre cada caso. Recibe
parámetros con nombres fijos (`inputs`, `outputs`, `expectations`, `trace`) y declaras solo
los que necesitas. El nombre de la función se convierte en el nombre de la métrica.

### Las tres métricas de citación

- **precision** — de lo que citaste, ¿cuánto era correcto? Castiga citar de más.
- **recall** — de lo que debías citar, ¿cuánto citaste? Castiga omitir evidencia.
- **F1** — media armónica. Es la que optimizamos, porque penaliza los extremos: citar todo
  (recall 1.0, precisión ínfima) y citar nada dan ambos F1 malo.

In [0]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback


def _prf(cited, gold) -> tuple[float, float, float]:
    """Precision, recall, F1 entre dos conjuntos de chunk_ids."""
    c, g = set(cited or []), set(gold or [])
    if not g:
        return 0.0, 0.0, 0.0
    inter = len(c & g)
    p = inter / len(c) if c else 0.0
    r = inter / len(g)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


@scorer
def citation_precision(outputs, expectations) -> Feedback:
    """De los chunks citados, qué fracción era realmente evidencia gold."""
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    p, _, _ = _prf(cited, gold)
    return Feedback(
        value=p,
        rationale=f"Citó {len(set(cited))} chunks, {len(set(cited) & set(gold))} correctos.",
    )


@scorer
def citation_recall(outputs, expectations) -> Feedback:
    """De la evidencia gold, qué fracción fue citada."""
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    _, r, _ = _prf(cited, gold)
    return Feedback(
        value=r,
        rationale=f"Recuperó {len(set(cited) & set(gold))} de {len(set(gold))} chunks gold.",
    )


@scorer
def citation_f1(outputs, expectations) -> float:
    """Media armónica de precision y recall de citación."""
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    return _prf(cited, gold)[2]


@scorer
def retrieval_hit(outputs, expectations) -> Feedback:
    """1.0 si al menos un chunk gold fue recuperado (aunque no se haya citado).

    Separa el fallo de retrieval del fallo de citación: si esto es 0, el problema está
    aguas arriba y ningún prompt lo va a arreglar.
    """
    retrieved = set((outputs or {}).get("retrieved_chunk_ids", []))
    gold = set((expectations or {}).get("gold_chunk_ids", []))
    hit = bool(retrieved & gold)
    return Feedback(
        value=1.0 if hit else 0.0,
        rationale="Evidencia gold presente en lo recuperado." if hit
                  else "Ningún chunk gold fue recuperado: falla de retrieval.",
    )

### Similitud de respuesta sin costo de LLM

Antes de gastar llamadas en un judge, una aproximación barata: F1 de tokens entre la
respuesta generada y la gold, al estilo de SQuAD. Es tosca (no entiende sinónimos) pero
gratis, y correlaciona razonablemente cuando las respuestas son cortas y factuales, como
aquí.

No sustituye al judge `Correctness`; sirve de red de seguridad si los judges no corren.

In [0]:
import re as _re


@scorer
def answer_token_f1(outputs, expectations) -> float:
    """F1 de tokens normalizados entre la respuesta y la gold (proxy barato de correctitud)."""
    def norm(t):
        return set(_re.findall(r"\w+", (t or "").lower()))

    pred = norm((outputs or {}).get("answer", ""))
    gold = norm((expectations or {}).get("expected_response", ""))
    if not pred or not gold:
        return 0.0
    inter = len(pred & gold)
    if inter == 0:
        return 0.0
    p, r = inter / len(pred), inter / len(gold)
    return 2 * p * r / (p + r)


DETERMINISTIC_SCORERS = [
    citation_precision,
    citation_recall,
    citation_f1,
    retrieval_hit,
    answer_token_f1,
]

print(f"Scorers determinísticos: {[s.name for s in DETERMINISTIC_SCORERS]}")

## 4. Evaluar el agente vectorial (capa A)

`mlflow.genai.evaluate()` ejecuta el agente sobre cada caso, captura la traza, corre los
scorers y registra todo como un run del experimento.

In [0]:
import mlflow

K_BASE = 3
PROMPT_BASE = "v1"
POLICY_BASE = "model"


def agent_fn(question: str) -> dict:
    """Envoltura del agente con la configuración base."""
    return rag_agent(
        question, k=K_BASE, prompt_version=PROMPT_BASE, citation_policy=POLICY_BASE
    )


with mlflow.start_run(run_name=f"eval_vector_k{K_BASE}_{PROMPT_BASE}_{POLICY_BASE}") as run_vec:
    mlflow.log_params({
        "retriever": "ai_search",
        "k": K_BASE,
        "prompt_version": PROMPT_BASE,
        "citation_policy": POLICY_BASE,
        "llm_endpoint": LLM_ENDPOINT,
        "n_eval": len(eval_data),
    })

    results_vector = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=agent_fn,
        scorers=DETERMINISTIC_SCORERS,
    )

print(f"\nRun registrado: {run_vec.info.run_id}")

### Resultados de la capa A

In [0]:
import pandas as pd


def show_metrics(results, titulo: str) -> pd.Series:
    """Imprime las métricas agregadas de un resultado de evaluación."""
    metrics = {k: v for k, v in results.metrics.items() if "/mean" in k}
    s = pd.Series(metrics).sort_index()
    print("=" * 56)
    print(f"  {titulo}")
    print("=" * 56)
    for k, v in s.items():
        print(f"  {k.replace('/mean', ''):<28} {v:.3f}")
    print("=" * 56)
    return s


m_vector = show_metrics(results_vector, f"AI SEARCH  k={K_BASE}  prompt={PROMPT_BASE}  pol={POLICY_BASE}")

## 5. Comparar contra la línea base TF-IDF

Misma muestra, mismos scorers, mismo prompt: lo único que cambia es el retriever. Así la
comparación es limpia y la diferencia es atribuible.

In [0]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from pyspark.ml import PipelineModel

N_FEATURES = 1 << 14
tfidf_model = PipelineModel.load(f"{VOL}/models/tfidf")

_rows = tfidf_model.transform(spark.table(T_CORPUS)).select(
    "chunk_id", "doc_id", "title", "doc_type", "chunk_text", "tfidf"
).collect()

_ids = [r["chunk_id"] for r in _rows]
_meta = {r["chunk_id"]: {"doc_id": r["doc_id"], "title": r["title"],
                         "doc_type": r["doc_type"], "chunk_text": r["chunk_text"]} for r in _rows}
_indptr, _indices, _data = [0], [], []
for r in _rows:
    v = r["tfidf"]
    _indices.extend(v.indices.tolist())
    _data.extend(v.values.tolist())
    _indptr.append(len(_indices))
_MATRIX = normalize(csr_matrix((_data, _indices, _indptr), shape=(len(_rows), N_FEATURES)),
                    norm="l2", axis=1)

print(f"Retriever TF-IDF reconstruido ({len(_ids):,} chunks).")

In [0]:
from mlflow.entities import Document, SpanType


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve_tfidf(question: str, k: int = 3) -> list[dict]:
    """Retriever TF-IDF, instrumentado igual que el vectorial para que los scorers lo lean."""
    q_vec = (tfidf_model.transform(spark.createDataFrame([(question,)], ["chunk_text"]))
             .select("tfidf").collect()[0]["tfidf"])
    q = normalize(
        csr_matrix((q_vec.values.tolist(), q_vec.indices.tolist(), [0, len(q_vec.indices)]),
                   shape=(1, N_FEATURES)),
        norm="l2",
    )
    scores = (_MATRIX @ q.T).toarray().ravel()
    chunks = [{"chunk_id": _ids[i], "score": float(scores[i]), **_meta[_ids[i]]}
              for i in np.argsort(-scores)[:k]]

    span = mlflow.get_current_active_span()
    if span is not None:
        span.set_outputs([
            Document(id=c["chunk_id"], page_content=c["chunk_text"],
                     metadata={"doc_uri": c["chunk_id"], "chunk_id": c["chunk_id"],
                               "relevance_score": c["score"]})
            for c in chunks
        ])
    return chunks


@mlflow.trace(name="rag_agent_tfidf")
def rag_agent_tfidf(question: str, k: int = 3, prompt_version: str = "v1",
                    citation_policy: str = "model") -> dict:
    chunks = retrieve_tfidf(question, k)
    answer, cited = generate(question, chunks, prompt_version)
    return {
        "answer": answer,
        "cited_chunk_ids": apply_citation_policy(cited, chunks, citation_policy),
        "retrieved_chunk_ids": [c["chunk_id"] for c in chunks],
    }

In [0]:
with mlflow.start_run(run_name=f"eval_baseline_tfidf_k{K_BASE}") as run_tfidf:
    mlflow.log_params({
        "retriever": "tfidf",
        "k": K_BASE,
        "prompt_version": PROMPT_BASE,
        "citation_policy": POLICY_BASE,
        "llm_endpoint": LLM_ENDPOINT,
        "n_eval": len(eval_data),
    })

    results_tfidf = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=lambda question: rag_agent_tfidf(
            question, k=K_BASE, prompt_version=PROMPT_BASE, citation_policy=POLICY_BASE
        ),
        scorers=DETERMINISTIC_SCORERS,
    )

m_tfidf = show_metrics(results_tfidf, f"TF-IDF  k={K_BASE}  prompt={PROMPT_BASE}  pol={POLICY_BASE}")

### Tabla comparativa

In [0]:
comp = pd.DataFrame({"tfidf": m_tfidf, "ai_search": m_vector})
comp.index = [i.replace("/mean", "") for i in comp.index]
comp["diferencia"] = comp["ai_search"] - comp["tfidf"]
comp["mejora_%"] = np.where(
    comp["tfidf"] > 0, (comp["diferencia"] / comp["tfidf"] * 100).round(1), np.nan
)

display(comp.reset_index().rename(columns={"index": "métrica"}))

Lee la tabla con cuidado. La pregunta útil no es "¿ganó el vectorial?" sino **"¿dónde
ganó?"**:

- Si `retrieval_hit` subió mucho pero `citation_precision` no se movió, el índice está
  trayendo mejor evidencia pero el agente sigue citando mal → el problema restante es el
  prompt o la política, no el retriever.
- Si `retrieval_hit` apenas cambió, el corpus tiene suficiente solapamiento léxico y TF-IDF
  ya era competitivo → el índice vectorial aporta poco *en este dataset*.

Con 20 preguntas, una diferencia de una o dos décimas puede ser ruido. Trátalo como señal
direccional, no como veredicto.

## 6. Capa B — LLM judges

Los scorers determinísticos comparan strings. No pueden saber si *"El servicio cayó por
agotamiento de conexiones"* y *"La interrupción se debió a saturación del pool de
conexiones"* dicen lo mismo. Para eso están los judges: modelos que evalúan semánticamente.

Tres judges, cada uno respondiendo una pregunta distinta:

| Judge | Pregunta que responde | Necesita gold |
|---|---|---|
| `Correctness` | ¿La respuesta coincide con la esperada? | Sí |
| `RetrievalGroundedness` | ¿La respuesta está respaldada por el contexto recuperado? | No |
| `RetrievalRelevance` | ¿Los chunks recuperados son relevantes a la pregunta? | No |

Los dos últimos leen el contexto **directamente del span RETRIEVER** de la traza. Por eso
importaba respetar el esquema de `Document` en el notebook 103.

`RetrievalGroundedness` es el detector de alucinaciones: mide si el modelo inventó algo que
no estaba en los chunks.

### Nota sobre cuota

Cada judge implica una llamada adicional al LLM por pregunta. Con 3 judges y 20 preguntas son
60 llamadas extra. Bajo uso justo esto puede activar throttling, así que reducimos la muestra
para esta capa y envolvemos todo en `try/except`: si falla, la capa A ya quedó registrada.

In [0]:
from mlflow.genai.scorers import Correctness, RetrievalGroundedness, RetrievalRelevance

N_JUDGE = min(10, len(eval_data))
judge_data = eval_data[:N_JUDGE]

print(f"Ejecutando judges sobre {N_JUDGE} preguntas...")

results_judges = None
try:
    with mlflow.start_run(run_name=f"eval_judges_k{K_BASE}_{PROMPT_BASE}") as run_judge:
        mlflow.log_params({
            "retriever": "ai_search", "k": K_BASE, "prompt_version": PROMPT_BASE,
            "citation_policy": POLICY_BASE, "n_eval": N_JUDGE, "layer": "llm_judges",
        })
        results_judges = mlflow.genai.evaluate(
            data=judge_data,
            predict_fn=agent_fn,
            scorers=[Correctness(), RetrievalGroundedness(), RetrievalRelevance()],
        )
    print("Judges completados.")
except Exception as e:
    print(
        f"\nLos judges no pudieron completarse: {type(e).__name__}: {e}\n\n"
        "Causa habitual: límites de uso justo de Free Edition.\n"
        "Las métricas determinísticas (capa A) ya quedaron registradas y son suficientes\n"
        "para continuar con los notebooks 105 y 106.\n"
        "Puedes reintentar más tarde o bajar N_JUDGE."
    )

In [0]:
if results_judges is not None:
    m_judges = show_metrics(results_judges, f"LLM JUDGES  (n={N_JUDGE})")
else:
    print("Sin resultados de judges en esta corrida.")

## 7. Análisis por caso

Los promedios esconden los casos interesantes. `mlflow.search_traces()` recupera el detalle
caso por caso, con las evaluaciones adjuntas.

In [0]:
traces = mlflow.search_traces(run_id=run_vec.info.run_id)
print(f"Trazas recuperadas: {len(traces)}")
display(traces.head(10))

### Los casos que fallaron

Reconstruimos el detalle para separar los dos modos de fallo. Esta distinción decide en qué
trabajas después:

- **Falla de retrieval** — la evidencia nunca llegó. Trabaja en el índice, en `k`, o en el
  chunking.
- **Falla de citación** — la evidencia llegó pero el agente no la citó bien. Trabaja en el
  prompt o en la política.

In [0]:
detalle = []
for case in eval_data:
    q = case["inputs"]["question"]
    gold = set(case["expectations"]["gold_chunk_ids"])
    out = agent_fn(q)
    cited, retrieved = set(out["cited_chunk_ids"]), set(out["retrieved_chunk_ids"])
    p, r, f1 = _prf(cited, gold)
    detalle.append({
        "question": q[:70],
        "gold": sorted(gold),
        "citado": sorted(cited),
        "hit_retrieval": bool(retrieved & gold),
        "precision": round(p, 2),
        "recall": round(r, 2),
        "f1": round(f1, 2),
    })
    time.sleep(0.3)

det_pd = pd.DataFrame(detalle)

fallos_retrieval = det_pd[~det_pd["hit_retrieval"]]
fallos_citacion = det_pd[det_pd["hit_retrieval"] & (det_pd["f1"] < 1.0)]
perfectos = det_pd[det_pd["f1"] == 1.0]

print(f"Casos perfectos (F1 = 1.0)            : {len(perfectos):>3} / {len(det_pd)}")
print(f"Fallas de RETRIEVAL (evidencia ausente): {len(fallos_retrieval):>3} / {len(det_pd)}")
print(f"Fallas de CITACIÓN (evidencia presente): {len(fallos_citacion):>3} / {len(det_pd)}")

In [0]:
display(det_pd.sort_values("f1"))

## 8. Explorar los resultados en la UI

Los números de este notebook están todos en la interfaz de MLflow, con más contexto.

**Comparar runs:**

1. Menú lateral → **Experiments** → `agenteval_rag`
2. En la lista de runs, marca las casillas de `eval_baseline_tfidf_k3` y
   `eval_vector_k3_v1_model`
3. Botón **Compare**
4. Las métricas quedan lado a lado, y los parámetros que difieren aparecen resaltados

**Ver el detalle de una evaluación:**

1. Abre un run → pestaña **Traces** (o **Evaluations**, según versión)
2. Cada fila es una pregunta, con sus scorers como columnas
3. Ordena por `citation_f1` ascendente para ver los peores casos primero
4. Clic en una fila → se abre la traza completa: qué se recuperó, qué prompt se envió,
   qué respondió el modelo
5. Si corrieron los judges, cada uno trae su **rationale**: el juez explica en texto por qué
   puntuó así

Ese rationale es lo más valioso para depurar. Un número te dice que algo va mal; el rationale
te dice qué.

## 9. Guardar el resumen

In [0]:
T_EVAL_SUMMARY = f"{CATALOG}.{SCHEMA}.agenteval_eval_summary"

resumen = comp.reset_index().rename(columns={"index": "metrica"})
resumen["evaluated_at"] = pd.Timestamp.utcnow().isoformat()
resumen["n_eval"] = len(eval_data)

(spark.createDataFrame(resumen)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T_EVAL_SUMMARY))

display(spark.table(T_EVAL_SUMMARY))